## loading

In [ ]:
!pip install transformers flash-linear-attention #causal_conv1d 

In [1]:
# loading-加载图像文本转文本模型（视觉语言模型）适用于接受图像输入的语言模型
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

model_dir='Qwen3.5-0.8B'
device='cuda'

model = AutoModelForImageTextToText.from_pretrained(
  model_dir,
  dtype=torch.bfloat16
).to(device)

# 初始化处理器
processor = AutoProcessor.from_pretrained(
  model_dir,
  trust_remote_code=True
)

/data/miniconda/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 473/473 [00:00<00:00, 4849.45it/s]


In [3]:
print(f"显存占用: {torch.cuda.memory_allocated(device)/1024**3:.3f} GB")

显存占用: 1.617 GB


In [4]:
# test-验证model功能
# 构建对话消息
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": "http://images.cocodataset.org/val2017/000000039769.jpg"},
            {"type": "text", "text": "在这张图片里你能看到什么？"},
        ]
    }
]
# 调用处理器processors应用聊天模板预处理其输出与图像输入
inputs = processor.tokenizer.apply_chat_template(
    messages, 
    add_generation_prompt=True, 
    tokenize=True, 
    return_dict=True, 
    return_tensors="pt"
).to(device)

# 前向传播，将预处理后的输入传递给模型
with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=200)

# 解码模型输出
input_len = len(inputs.input_ids[0])
generated_texts = processor.batch_decode(generated_ids[:, input_len:], skip_special_tokens=True)
print(generated_texts)

[transformers] Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


['这张图片里主要包含以下内容：\n\n1.  **文字**：图片上方有一行中文文字，内容是“在这张图片里你能看到什么？”。\n2.  **背景**：背景是纯白色的。\n\n所以，这张图片就是一个简单的文字说明，没有具体的图像内容。\n']


## visit .visual

In [4]:
# manufacture-构造960x540的Image
from PIL import Image

# 1. 加载JPG
img = Image.open("微信图片_20260327210311_589_2.jpg")
# 2. 确保RGB 3通道
if img.mode != 'RGB':
    img = img.convert('RGB')
# 3. 调整尺寸为 960x540
img = img.resize((960, 540), Image.LANCZOS)
visual_input = processor.image_processor(images=[img]).to(device)
visual_input.pixel_values.shape

torch.Size([2040, 1536])

In [6]:
visual_input.pixel_values

tensor([[ 0.2549,  0.2549,  0.2549,  ...,  0.8039,  0.8039,  0.8039],
        [ 0.2549,  0.2549,  0.2706,  ...,  0.8118,  0.8118,  0.8118],
        [ 0.2471,  0.2471,  0.2471,  ...,  0.8196,  0.8196,  0.8196],
        ...,
        [-0.2627, -0.2471, -0.2471,  ..., -0.7490, -0.7804, -0.7882],
        [-0.4588, -0.4196, -0.3569,  ..., -0.5529, -0.5373, -0.4980],
        [-0.4980, -0.5294, -0.5529,  ..., -0.5059, -0.4980, -0.4902]],
       device='cuda:0')

## reform .visual.forward()

In [2]:
import torch
import torch.nn.functional as F
from transformers.modeling_outputs import BaseModelOutputWithPooling
import types

def visual_stage_1_forward(self, hidden_states: torch.Tensor, grid_thw: torch.Tensor, 
                          stop_layer_idx: int = None, **kwargs):
    """
    覆写 Qwen3.5 视觉头 forward，支持：
    - 输入维度兼容：torch.Size([S, D]) 或 torch.Size([B, S, D])
    - stop_layer_idx: 运行到指定层（包含该层），例如 8 表示运行第 0~8 层
    - return_layer_indices: 需要返回中间输出的层索引列表（预留）
    """
    total_layers = len(self.blocks)
    if stop_layer_idx is None:
        stop_layer_idx = total_layers - 1

    # ✅ 1. 记录原始维度并统一展平为 2D，兼容 [B, S, D] 输入
    is_3d = hidden_states.dim() == 3
    if is_3d:
        B, S, D = hidden_states.shape
        hidden_states = hidden_states.view(-1, D)  # 展平为 (B*S, D)
    else:
        B, S, D = None, hidden_states.shape[0], hidden_states.shape[1]

    # 2. Patch Embedding
    hidden_states = self.patch_embed(hidden_states)

    # 3. Position Embedding
    pos_embeds = self.fast_pos_embed_interpolate(grid_thw)
    hidden_states = hidden_states + pos_embeds

    # 4. Rotary Position Embedding
    rotary_pos_emb = self.rot_pos_emb(grid_thw)
    seq_len = hidden_states.shape[0]  # ✅ 安全获取长度，避免 3D 解包报错
    hidden_states = hidden_states.reshape(seq_len, -1)
    rotary_pos_emb = rotary_pos_emb.reshape(seq_len, -1)
    emb = torch.cat((rotary_pos_emb, rotary_pos_emb), dim=-1)
    position_embeddings = (emb.cos(), emb.sin())

    # 5. 准备变长注意力所需的 cu_seqlens
    cu_seqlens = torch.repeat_interleave(grid_thw[:, 1] * grid_thw[:, 2], grid_thw[:, 0]).cumsum(
        dim=0, dtype=grid_thw.dtype if torch.jit.is_tracing() else torch.int32
    )
    cu_seqlens = F.pad(cu_seqlens, (1, 0), value=0)

    # 6. 逐层 Forward 并截取指定深度
    intermediate_outputs = {}
    for idx, blk in enumerate(self.blocks):
        hidden_states = blk(
            hidden_states,
            cu_seqlens=cu_seqlens,
            position_embeddings=position_embeddings,
            **kwargs,
        )
        # 达到指定深度后提前终止
        if idx >= stop_layer_idx:
            break

    # 7. 最终特征融合（Merger）
    merged_hidden_states = self.merger(hidden_states)

    # ✅ 若原始输入为 3D，则按 Batch 维度恢复形状
    if is_3d:
        # merger 可能会改变 token 数量（如 pooling），需动态计算新序列长度
        new_seq_len = hidden_states.shape[0] // B
        hidden_states = hidden_states.view(B, new_seq_len, -1)
        
        # pooler_output 通常也按 batch 排列，尝试对齐恢复
        if merged_hidden_states.dim() == 2 and merged_hidden_states.shape[0] % B == 0:
            merged_seq_len = merged_hidden_states.shape[0] // B
            merged_hidden_states = merged_hidden_states.view(B, merged_seq_len, -1)

    return BaseModelOutputWithPooling(
        last_hidden_state=hidden_states,
        pooler_output=merged_hidden_states,
    )
model.model.visual.visual_stage_1_forward = types.MethodType(visual_stage_1_forward, model.model.visual)

In [24]:
#test-visual_stage_1_forward
v_1_output = model.model.visual.visual_stage_1_forward(
    hidden_states=visual_input.pixel_values.unsqueeze(0), 
    grid_thw=visual_input.image_grid_thw, 
    stop_layer_idx=7, 
)
v_1_output

BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 0.1055,  0.1387,  0.2207,  ...,  1.0938,  0.4414,  0.5039],
         [ 0.2520, -0.0552,  0.2207,  ...,  0.6055,  0.3086,  0.4453],
         [ 0.2207,  0.0977,  0.3516,  ...,  0.3320,  0.2002,  0.5078],
         ...,
         [-0.2480,  0.2090, -0.0096,  ...,  0.2148, -0.2891,  0.0664],
         [-0.0825,  0.0942, -0.1699,  ..., -0.2070,  0.0430,  0.0796],
         [-0.1270,  0.0148, -0.0396,  ...,  0.1816,  0.0908,  0.2148]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<ViewBackward0>), pooler_output=tensor([[[ 7.2812, -0.7227, -0.9766,  ...,  0.9727,  0.2256, -0.9414],
         [ 0.6836, -0.3555, -0.2812,  ...,  0.8242, -0.7461, -0.9766],
         [ 6.6250, -0.9805, -1.1406,  ...,  0.6250, -0.2812, -1.0000],
         ...,
         [ 0.2236, -0.3066, -0.2852,  ..., -0.0713, -0.9531,  0.7109],
         [ 6.3125, -0.7852, -0.6406,  ..., -0.4512, -1.2812,  0.3770],
         [12.8750, -0.2793, -1.4531,  ..., -0.2402, -0.263

In [7]:
v_1_output.last_hidden_state.shape, v_1_output.pooler_output.shape

(torch.Size([1, 2040, 768]), torch.Size([1, 510, 1024]))

In [3]:
import types
import torch
import torch.nn.functional as F
from typing import Optional, Tuple
from transformers.modeling_outputs import BaseModelOutputWithPooling

def visual_stage_2_forward(
    self,
    hidden_states: torch.Tensor,
    start_layer: int = 2,
    end_layer: int = 8,
    cu_seqlens: Optional[torch.Tensor] = None,
    position_embeddings: Optional[Tuple[torch.Tensor, torch.Tensor]] = None,
    grid_thw: Optional[torch.Tensor] = None
) -> BaseModelOutputWithPooling:  # ✅ 修正返回类型注解以匹配实际返回对象
    """
    仅推理视觉编码器的指定深度范围。
    兼容输入维度：torch.Size([S, D]) 或 torch.Size([B, S, D])
    """
    end_layer = min(end_layer, len(self.blocks))
    if start_layer >= end_layer:
        raise ValueError("start_layer 必须小于 end_layer")

    # ✅ 1. 维度检测与统一展平为 2D
    is_3d = hidden_states.dim() == 3
    if is_3d:
        B, S, D = hidden_states.shape
        hidden_states = hidden_states.view(-1, D)  # 展平为 [Total_Seq, D]
    else:
        B = None

    # 若未提供注意力所需的辅助张量，则通过 grid_thw 自动计算
    if cu_seqlens is None or position_embeddings is None:
        if grid_thw is None:
            raise ValueError("未提供 cu_seqlens/position_embeddings 时，必须传入 grid_thw")
        seq_len = hidden_states.shape[0]  # ✅ 修复：统一使用展平后的序列长度，避免 3D 解包报错

        # 1. 计算 cu_seqlens (用于变长序列注意力)
        cu_seqlens = torch.repeat_interleave(grid_thw[:, 1] * grid_thw[:, 2], grid_thw[:, 0]).cumsum(
            dim=0, dtype=torch.int32
        )
        cu_seqlens = F.pad(cu_seqlens, (1, 0), value=0).to(hidden_states.device)

        # 2. 计算 position_embeddings (RoPE 的 cos/sin)
        rotary_pos_emb = self.rot_pos_emb(grid_thw).reshape(seq_len, -1)
        emb = torch.cat((rotary_pos_emb, rotary_pos_emb), dim=-1)
        position_embeddings = (emb.cos(), emb.sin())

    # 逐层前向传播
    for idx in range(start_layer, end_layer):
        blk = self.blocks[idx]
        hidden_states = blk(
            hidden_states,
            cu_seqlens=cu_seqlens,
            position_embeddings=position_embeddings
        )

    merged_hidden_states = self.merger(hidden_states)

    # ✅ 2. 恢复 Batch 维度（若原始输入为 3D）
    if is_3d:
        # 动态计算当前序列长度，兼容 merger 可能改变 token 数量的情况
        new_seq_len = hidden_states.shape[0] // B
        hidden_states = hidden_states.view(B, new_seq_len, -1)

        # pooler_output 维度对齐（仅当能被 B 整除时恢复 3D，否则保持 2D pooled 特征）
        if merged_hidden_states.dim() == 2 and merged_hidden_states.shape[0] % B == 0:
            merged_seq_len = merged_hidden_states.shape[0] // B
            merged_hidden_states = merged_hidden_states.view(B, merged_seq_len, -1)

    return BaseModelOutputWithPooling(
        last_hidden_state=hidden_states,
        pooler_output=merged_hidden_states,
    )
# 🌟 关键修复：使用 types.MethodType 正确绑定 self
model.model.visual.visual_stage_2_forward = types.MethodType(visual_stage_2_forward, model.model.visual)

In [9]:
v_2_output = model.model.visual.visual_stage_2_forward(
        hidden_states=v_1_output.last_hidden_state,
        start_layer=2,
        end_layer=8,
        grid_thw=visual_input.image_grid_thw
    )
v_2_output

BaseModelOutputWithPooling(last_hidden_state=tensor([[[ 0.2441,  0.1562,  0.3926,  ...,  0.9648,  0.3125,  0.2559],
         [ 0.2354,  0.1006,  0.3262,  ...,  0.7148,  0.2930,  0.2236],
         [ 0.2832,  0.1807,  0.4238,  ...,  0.4004,  0.3301,  0.2256],
         ...,
         [ 0.0088, -0.0371,  0.0052,  ..., -0.0225, -0.0811, -0.1738],
         [ 0.0339,  0.0186,  0.1025,  ..., -0.0151,  0.0359, -0.0684],
         [ 0.1162,  0.0383,  0.1641,  ...,  0.3164,  0.1216,  0.0405]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<ViewBackward0>), pooler_output=tensor([[[ 1.3875e+01, -7.1094e-01, -2.1562e+00,  ...,  1.4219e+00,
           7.0703e-01, -8.0078e-01],
         [ 9.3125e+00, -4.5508e-01, -1.5156e+00,  ...,  1.2422e+00,
          -1.2665e-03, -1.1719e+00],
         [ 8.5000e+00, -8.1641e-01, -1.8672e+00,  ...,  6.5234e-01,
          -1.7773e-01, -8.7109e-01],
         ...,
         [ 6.4375e+00,  1.2695e-01, -2.1562e+00,  ...,  4.9609e-01,
          -4.6680e-01, -3.6523

In [10]:
v_2_output.last_hidden_state.shape

torch.Size([1, 2040, 768])

In [ ]:
# define-language_stage_1_forward
import torch
import types
import sys
from transformers.cache_utils import DynamicCache
from transformers.masking_utils import create_causal_mask
Qwen3_5ModelOutputWithPast = sys.modules[model.__class__.__module__].Qwen3_5ModelOutputWithPast

def language_stage_1_forward(
    self, 
    inputs_embeds: torch.Tensor, 
    depth: int, 
    attention_mask: torch.Tensor | None = None, 
    past_key_values: DynamicCache | None = None, 
    use_cache: bool = True
) -> "Qwen3_5ModelOutputWithPast":
    assert 0 < depth <= len(self.layers), f"depth 必须在 1 到 {len(self.layers)} 之间"
    
    if use_cache and past_key_values is None:
        past_key_values = DynamicCache(config=self.config)
        
    batch_size, seq_len, _ = inputs_embeds.shape
    device = inputs_embeds.device
    past_seen = past_key_values.get_seq_length() if past_key_values is not None else 0
    
    position_ids = torch.arange(seq_len, device=device) + past_seen
    position_ids = position_ids.view(1, 1, -1).expand(4, batch_size, -1)
    
    text_position_ids = position_ids[0]
    position_ids_for_rope = position_ids[1:]  
    
    causal_mask = create_causal_mask(
        config=self.config,
        inputs_embeds=inputs_embeds,
        attention_mask=None,#attention_mask,
        past_key_values=past_key_values,
        position_ids=text_position_ids,
    )
    
    try:
        linear_attn_mask = self._update_linear_attn_mask(attention_mask, past_key_values)
    except (ValueError, StopIteration):
        linear_attn_mask = attention_mask
        
    position_embeddings = self.rotary_emb(inputs_embeds, position_ids_for_rope)
    
    hidden_states = inputs_embeds
    for i in range(depth):
        layer_type = self.config.layer_types[i]
        layer_mask = linear_attn_mask if layer_type == "linear_attention" else causal_mask
        
        hidden_states = self.layers[i](
            hidden_states,
            position_embeddings=position_embeddings,
            attention_mask=layer_mask,
            position_ids=text_position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
        )
    
    # ⚠️ 若此函数为完整前向传播的终点，请取消下一行注释
    # hidden_states = self.norm(hidden_states)
        
    return Qwen3_5ModelOutputWithPast(
        last_hidden_state=hidden_states,
        past_key_values=past_key_values if use_cache else None,
    )

# 绑定到 language_model 实例
model.model.language_model.language_stage_1_forward = types.MethodType(language_stage_1_forward, model.model.language_model)

In [29]:
# test-language_stage_1_forward
# # 1. 准备输入 (与你提供的维度一致)
inputs_embeds = torch.randn(1, 4, 1024, device=model.device, dtype=model.dtype)

# 2. 仅推理前 5 层，获取第 5 层输出
depth = 1
l_1_outputs = model.model.language_model.language_stage_1_forward(inputs_embeds, depth=depth)
l_1_outputs

Qwen3_5ModelOutputWithPast(last_hidden_state=tensor([[[-0.5859,  0.9688,  0.4473,  ..., -0.4297,  0.9609, -0.7266],
         [ 0.3789,  0.4512, -0.1992,  ...,  0.4141, -0.0176,  0.7773],
         [-0.5430, -1.5547, -0.0898,  ..., -0.1426,  0.0796,  1.2344],
         [-1.7578,  0.0723,  0.3340,  ..., -0.9609,  1.4297, -0.9453]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<AddBackward0>), past_key_values=DynamicCache(layers=[LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer]), hidden_states=None, attentions=None, rope_deltas=None)

In [13]:
l_1_outputs.last_hidden_state.shape

torch.Size([1, 4, 1024])

In [5]:
def language_stage_2_forward(
    self,
    hidden_states: torch.Tensor,
    start_layer: int = 2,
    end_layer: int = 8,
    attention_mask: torch.Tensor | None = None,
    past_key_values=None,
    use_cache: bool = True,
    **kwargs
):
    # ✅ 关键修复：首次调用时初始化 DynamicCache
    if use_cache and past_key_values is None:
        past_key_values = DynamicCache(config=self.config)
        
    device = hidden_states.device
    batch_size, seq_len, _ = hidden_states.shape

    # 基于 KV Cache 长度计算绝对位置偏移
    past_seen = past_key_values.get_seq_length()
    position_ids = torch.arange(past_seen, past_seen + seq_len, device=device)
    position_ids = position_ids.view(1, batch_size, -1).expand(3, batch_size, -1)

    # 计算 RoPE (cos, sin)
    position_embeddings = self.rotary_emb(hidden_states, position_ids)

    # 生成注意力掩码
    if attention_mask is None:
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool))
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)
        linear_attn_mask = None
    else:
        causal_mask = attention_mask
        linear_attn_mask = attention_mask

    num_layers = len(self.layers)
    if not (0 <= start_layer <= end_layer < num_layers):
        raise ValueError(f"Layer index out of range. Available: 0~{num_layers-1}")

    for i in range(start_layer, end_layer + 1):
        layer = self.layers[i]
        layer_type = self.config.layer_types[i]
        layer_mask = linear_attn_mask if layer_type == "linear_attention" else causal_mask

        hidden_states = layer(
            hidden_states=hidden_states,
            position_embeddings=position_embeddings,
            attention_mask=layer_mask,
            position_ids=position_ids[0],
            past_key_values=past_key_values,
            use_cache=use_cache,
            **kwargs
        )

    return Qwen3_5ModelOutputWithPast(
        last_hidden_state=hidden_states,
        past_key_values=past_key_values,
    )
model.model.language_model.language_stage_2_forward = types.MethodType(
    language_stage_2_forward, 
    model.model.language_model
)

In [ ]:
import torch
import types
from transformers.cache_utils import DynamicCache
from transformers.masking_utils import create_causal_mask

# 确保能正确引用 Output 类（根据你的环境调整模块路径）
Qwen3_5ModelOutputWithPast = sys.modules[model.__class__.__module__].Qwen3_5ModelOutputWithPast

def language_stage_2_forward(
    self,
    hidden_states: torch.Tensor,
    start_layer: int = 2,
    end_layer: int = 8,
    attention_mask: torch.Tensor | None = None,  # ⚠️ 流式模式下此参数将被安全忽略
    past_key_values=None,
    use_cache: bool = True,
    **kwargs
):
    # 1. 初始化/获取 Cache
    if use_cache and past_key_values is None:
        past_key_values = DynamicCache(config=self.config)

    device = hidden_states.device
    batch_size, seq_len, _ = hidden_states.shape
    past_seen = past_key_values.get_seq_length()
    total_len = past_seen + seq_len

    # 2. 构造 4D Position IDs (严格对齐 Qwen3.5 MRoPE)
    position_ids = torch.arange(past_seen, total_len, device=device)
    position_ids = position_ids.view(1, batch_size, -1).expand(4, batch_size, -1)  # (4, B, S)
    text_position_ids = position_ids[0]      # (B, S) 供 DecoderLayer 使用
    rope_position_ids = position_ids[1:]     # (3, B, S) 供 RotaryEmbedding 使用

    # 3. 🔥 核心修复：手动构造增量 Causal Mask，彻底避开 create_causal_mask 的广播坑
    # SDPA 期望形状: [Batch, 1, Query_Seq, Key_Total]
    # True = 允许注意力, False = 遮蔽未来 Token
    row_idx = torch.arange(seq_len, device=device).unsqueeze(1)      # [S, 1]
    col_idx = torch.arange(total_len, device=device)                 # [K_total]
    causal_mask = col_idx <= row_idx + past_seen                     # [S, K_total] 布尔矩阵
    causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)              # [1, 1, S, K_total]

    # 4. Linear Attention Mask 处理 (Decode 阶段直接置 None 依赖 RNN State)
    linear_attn_mask = None
    if past_seen == 0:  # 仅 Prefill 阶段需要
        try:
            linear_attn_mask = self._update_linear_attn_mask(attention_mask, past_key_values)
        except Exception:
            linear_attn_mask = attention_mask

    # 5. 计算 RoPE (cos, sin)
    position_embeddings = self.rotary_emb(hidden_states, rope_position_ids)

    # 6. 逐层 Forward
    for i in range(start_layer, end_layer + 1):
        layer = self.layers[i]
        layer_type = self.config.layer_types[i]
        # 动态分配 Mask
        layer_mask = linear_attn_mask if layer_type == "linear_attention" else causal_mask

        hidden_states = layer(
            hidden_states=hidden_states,
            position_embeddings=position_embeddings,
            attention_mask=layer_mask,
            position_ids=text_position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            **kwargs
        )

    return Qwen3_5ModelOutputWithPast(
        last_hidden_state=hidden_states,
        past_key_values=past_key_values if use_cache else None,
    )

# 🔗 重新绑定修正后的方法
model.model.language_model.language_stage_2_forward = types.MethodType(
    language_stage_2_forward, model.model.language_model
)

In [31]:
# test-language_stage_2_forward
l_2_outputs = model.model.language_model.language_stage_2_forward(
    hidden_states=l_1_outputs.last_hidden_state,
    start_layer=1,  # 对应第2层
    end_layer=23     # 对应第24层
)

l_2_outputs

Qwen3_5ModelOutputWithPast(last_hidden_state=tensor([[[-0.9141,  0.0264,  1.4531,  ..., -1.1797,  1.1172, -0.4844],
         [ 5.8125,  0.3164, -0.1855,  ...,  0.8906, -0.2197,  0.6211],
         [-0.2598, -2.5312,  0.7266,  ...,  0.2695, -0.4082,  1.7031],
         [ 0.4141,  0.4414,  0.9062,  ..., -1.0781,  1.9219, -1.9688]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<AddBackward0>), past_key_values=DynamicCache(layers=[LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer]), hidden_states=None, attentions=None, rope_deltas=None)

In [32]:
l_2_outputs.last_hidden_state.shape

torch.Size([1, 4, 1024])

## cross-attention

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
Qwen3_5RMSNorm = sys.modules[model.__class__.__module__].Qwen3_5RMSNorm

class Qwen3_5CrossAttention(nn.Module):
    def __init__(self, config, vis_hidden_size=768):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_key_value_heads = config.num_key_value_heads
        self.head_dim = self.hidden_size // self.num_attention_heads
        self.scaling = self.head_dim ** -0.5

        # 维度对齐层
        self.vis_align_proj = nn.Linear(vis_hidden_size, self.hidden_size, bias=config.attention_bias) \
            if vis_hidden_size != self.hidden_size else nn.Identity()

        # Q 来自语言，K/V 来自视觉
        self.q_proj = nn.Linear(self.hidden_size, self.num_attention_heads * self.head_dim, bias=config.attention_bias)
        self.k_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=config.attention_bias)
        self.v_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=config.attention_bias)
        self.o_proj = nn.Linear(self.num_attention_heads * self.head_dim, self.hidden_size, bias=config.attention_bias)

        # Qwen3.5 特有的 Head-wise RMSNorm
        self.q_norm = Qwen3_5RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = Qwen3_5RMSNorm(self.head_dim, eps=config.rms_norm_eps)

    def forward(self, lang_hidden_states: torch.Tensor, vis_hidden_states: torch.Tensor, attention_mask: torch.Tensor = None):
        # 兼容 2D 输入 [seq, vis_dim] -> [1, seq, vis_dim]
        if vis_hidden_states.dim() == 2:
            vis_hidden_states = vis_hidden_states.unsqueeze(0)
            
        # 🔑 关键修复：强制将视觉特征转换到与投影层权重一致的 dtype（Half/BFloat16）
        vis_hidden_states = vis_hidden_states.to(self.vis_align_proj.weight.dtype)

        # 视觉特征维度对齐
        vis_hidden_states = self.vis_align_proj(vis_hidden_states)

        batch_size, seq_len, _ = lang_hidden_states.shape
        vis_seq_len = vis_hidden_states.shape[1]

        # Query (Language)
        query_states = self.q_proj(lang_hidden_states)
        query_states = query_states.view(batch_size, seq_len, self.num_attention_heads, self.head_dim)
        query_states = self.q_norm(query_states).transpose(1, 2)

        # Key & Value (Vision)
        key_states = self.k_proj(vis_hidden_states)
        value_states = self.v_proj(vis_hidden_states)
        key_states = key_states.view(batch_size, vis_seq_len, self.num_key_value_heads, self.head_dim)
        key_states = self.k_norm(key_states).transpose(1, 2)
        value_states = value_states.view(batch_size, vis_seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        # GQA 扩展
        if self.num_key_value_heads != self.num_attention_heads:
            n_rep = self.num_attention_heads // self.num_key_value_heads
            key_states = key_states.repeat_interleave(n_rep, dim=1)
            value_states = value_states.repeat_interleave(n_rep, dim=1)

        # Scaled Dot-Product Attention
        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) * self.scaling
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask

        attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_output = torch.matmul(attn_weights, value_states)

        # 还原维度并投影回语言空间
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        attn_output = self.o_proj(attn_output)
        return attn_output

class Qwen3_5VisualInjector(nn.Module):
    def __init__(self, config, vis_hidden_size=768):
        super().__init__()
        self.cross_attn = Qwen3_5CrossAttention(config, vis_hidden_size)
        self.post_attn_norm = Qwen3_5RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, lang_hidden_states, vis_hidden_states, attention_mask=None):
        attn_out = self.cross_attn(lang_hidden_states, vis_hidden_states, attention_mask)
        return self.post_attn_norm(lang_hidden_states + attn_out)

In [35]:
# 1. 初始化注入模块 (使用 Text 部分的配置)
text_config = model.config.text_config
visual_injector = Qwen3_5VisualInjector(text_config, vis_hidden_size=768)
visual_injector.to(model.device, dtype=model.dtype)
visual_injector.train()  # 或 .eval() 根据场景决定

# 3. 执行注入
updated_lang_states = visual_injector(l_outputs.last_hidden_state, v_1_output.last_hidden_state)

print(updated_lang_states.shape)  # 输出: torch.Size([1, 100, 2048])，视觉语义已融合

torch.Size([1, 5749, 1024])


In [8]:
GradientCheckpointingLayer = sys.modules[model.__class__.__module__].GradientCheckpointingLayer  # 根据实际路径调整
Qwen3_5MLP = sys.modules[model.__class__.__module__].Qwen3_5MLP

class Qwen3_5VisionToTextBlock(GradientCheckpointingLayer):
    """
    Vision-to-Text Cross-Attention Block.
    
    Allows text tokens to attend to vision tokens via cross-attention,
    followed by a language-side MLP. Follows the pre-norm residual 
    architecture similar to Qwen3_5VisionBlock.
    
    Input:
        lang_hidden_states: [batch, text_seq, hidden_size]  e.g., [1, 100, 1024]
        vis_hidden_states:  [vis_seq, vis_hidden_size]      e.g., [2040, 768]
                          or [batch, vis_seq, vis_hidden_size]
    
    Output:
        updated lang_hidden_states: [batch, text_seq, hidden_size]
    """
    
    def __init__(self, config, vis_hidden_size: int = 768, attn_implementation: str = "sdpa") -> None:
        super().__init__()
        self.hidden_size = config.hidden_size
        
        # === Cross-Attention Branch (Vision → Text) ===
        # Pre-norm for language query (Qwen3.5 style RMSNorm)
        self.lang_norm1 = Qwen3_5RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        
        # Cross-attention: Q from language, K/V from vision
        self.cross_attn = Qwen3_5CrossAttention(config, vis_hidden_size=vis_hidden_size)
        
        # Post-norm after cross-attn residual connection
        self.lang_norm2 = Qwen3_5RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        
        # === MLP Branch (Language-side FFN) ===
        # Pre-norm for MLP
        self.lang_norm3 = Qwen3_5RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        
        # Language MLP (SwiGLU style, from Qwen3_5MLP)
        self.mlp = Qwen3_5MLP(config, config.intermediate_size)

    def forward(
        self,
        lang_hidden_states: torch.Tensor,      # [batch, text_seq, hidden_size]
        vis_hidden_states: torch.Tensor,       # [vis_seq, vis_hidden_size] or [batch, vis_seq, vis_hidden_size]
        attention_mask: torch.Tensor | None = None,
        **kwargs,
    ) -> torch.Tensor:
        # ─── Cross-Attention: Text attends to Vision ─────────────────────────
        residual = lang_hidden_states
        lang_normed = self.lang_norm1(lang_hidden_states)
        
        attn_output = self.cross_attn(
            lang_hidden_states=lang_normed,
            vis_hidden_states=vis_hidden_states,
            attention_mask=attention_mask,
        )
        # Residual + post-norm (Qwen3.5 pattern)
        lang_hidden_states = self.lang_norm2(residual + attn_output)
        
        # ─── MLP: Language-side Feed-Forward ─────────────────────────────────
        residual = lang_hidden_states
        lang_normed = self.lang_norm3(lang_hidden_states)
        
        mlp_output = self.mlp(lang_normed)
        # Residual connection (standard Transformer pattern)
        lang_hidden_states = residual + mlp_output
        
        return lang_hidden_states

In [38]:
# model.config.text_config, model.model.language_model.config
v_to_l_block = Qwen3_5VisionToTextBlock(
    model.config.text_config, 
    vis_hidden_size=768).to(model.device, dtype=model.dtype)

l_outputs.last_hidden_state, v_1_output.last_hidden_state

updated_text = v_to_l_block(
    lang_hidden_states=l_outputs.last_hidden_state,
    vis_hidden_states=v_1_output.last_hidden_state
)  # → [1, 100, 1024]
updated_text

tensor([[[ 3.2656,  0.5469, -1.7422,  ...,  1.6406,  1.0156, -0.5938],
         [-0.7734, -0.5547, -1.1562,  ...,  1.8438,  0.2246,  0.3125],
         [-1.1172, -0.5469, -1.5625,  ...,  1.0938,  1.2266,  0.5000],
         ...,
         [ 1.9766,  2.0000, -0.4336,  ...,  1.7031,  0.5664, -0.8086],
         [ 2.3125,  0.8477, -0.4023,  ...,  2.4219,  0.4023,  0.3867],
         [ 2.2500, -0.2969, -0.4805,  ...,  0.7969,  0.2158, -0.7070]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<AddBackward0>)

In [9]:
class Qwen3_5LangToVisionCrossAttention(nn.Module):
    def __init__(self, vis_hidden_size=768, lang_hidden_size=1024,
                 num_attention_heads=12, num_key_value_heads=4,
                 rms_norm_eps=1e-6, attention_bias=False):
        super().__init__()
        self.vis_hidden_size = vis_hidden_size
        self.lang_hidden_size = lang_hidden_size
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.head_dim = vis_hidden_size // num_attention_heads  # 768 // 12 = 64
        self.scaling = self.head_dim ** -0.5
        self.num_key_value_groups = num_attention_heads // num_key_value_heads

        # Q 来自视觉，K/V 来自语言
        self.q_proj = nn.Linear(vis_hidden_size, num_attention_heads * self.head_dim, bias=attention_bias)
        self.k_proj = nn.Linear(lang_hidden_size, num_key_value_heads * self.head_dim, bias=attention_bias)
        self.v_proj = nn.Linear(lang_hidden_size, num_key_value_heads * self.head_dim, bias=attention_bias)
        self.o_proj = nn.Linear(num_attention_heads * self.head_dim, vis_hidden_size, bias=attention_bias)

        # Qwen3.5 特有的 Head-wise RMSNorm (作用在 head_dim 维度)
        self.q_norm = Qwen3_5RMSNorm(self.head_dim, eps=rms_norm_eps)
        self.k_norm = Qwen3_5RMSNorm(self.head_dim, eps=rms_norm_eps)

    def forward(self, vis_hidden_states, lang_hidden_states, attention_mask=None):
        # 兼容 2D 输入 [seq, dim] -> [1, seq, dim]
        if vis_hidden_states.dim() == 2: vis_hidden_states = vis_hidden_states.unsqueeze(0)
        if lang_hidden_states.dim() == 2: lang_hidden_states = lang_hidden_states.unsqueeze(0)

        # 🔑 关键修复：动态对齐输入 dtype 与投影层权重 dtype (Half / BFloat16 / Float)
        target_dtype = self.q_proj.weight.dtype
        vis_hidden_states = vis_hidden_states.to(target_dtype)
        lang_hidden_states = lang_hidden_states.to(target_dtype)

        batch_size, vis_seq_len, _ = vis_hidden_states.shape
        lang_seq_len = lang_hidden_states.shape[1]

        # Query (Vision)
        query_states = self.q_proj(vis_hidden_states).view(batch_size, vis_seq_len, self.num_attention_heads, self.head_dim)
        query_states = self.q_norm(query_states).transpose(1, 2)  # (bs, n_heads, vis_seq, head_dim)

        # Key & Value (Language)
        key_states = self.k_proj(lang_hidden_states).view(batch_size, lang_seq_len, self.num_key_value_heads, self.head_dim)
        key_states = self.k_norm(key_states).transpose(1, 2)    # (bs, n_kv_heads, lang_seq, head_dim)
        value_states = self.v_proj(lang_hidden_states).view(batch_size, lang_seq_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        # GQA 扩展 (与 Qwen3.5 保持一致)
        if self.num_key_value_groups > 1:
            key_states = key_states.repeat_interleave(self.num_key_value_groups, dim=1)
            value_states = value_states.repeat_interleave(self.num_key_value_groups, dim=1)

        # Scaled Dot-Product Attention
        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3)) * self.scaling
        if attention_mask is not None:
            # 期望 attention_mask 形状: (batch, 1, vis_seq, lang_seq)
            attn_weights = attn_weights + attention_mask

        attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query_states.dtype)
        attn_output = torch.matmul(attn_weights, value_states)

        # 还原维度并投影回视觉空间
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, vis_seq_len, -1)
        return self.o_proj(attn_output)

class Qwen3_5VisionCrossInjector(nn.Module):
    """包含残差连接与后归一化的注入包装器"""
    def __init__(self, vis_hidden_size=768, lang_hidden_size=1024,
                 num_attention_heads=12, num_key_value_heads=4, rms_norm_eps=1e-6):
        super().__init__()
        self.cross_attn = Qwen3_5LangToVisionCrossAttention(vis_hidden_size, lang_hidden_size,
                                                            num_attention_heads, num_key_value_heads, rms_norm_eps)
        self.post_attn_norm = Qwen3_5RMSNorm(vis_hidden_size, eps=rms_norm_eps)

    def forward(self, vis_hidden_states, lang_hidden_states, attention_mask=None):
        attn_out = self.cross_attn(vis_hidden_states, lang_hidden_states, attention_mask)
        # 残差连接 + Post-Norm (特征融合常用稳定结构)
        return self.post_attn_norm(vis_hidden_states + attn_out)

In [39]:
visual_cross_injector = Qwen3_5VisionCrossInjector(
    vis_hidden_size=768, 
    lang_hidden_size=1024,
    num_attention_heads=12,
    num_key_value_heads=4
)
# 🌟 强制与主模型设备与精度同步，避免 dtype 报错
visual_cross_injector = visual_cross_injector.to(device=model.device, dtype=model.dtype)

updated_vis_states = visual_cross_injector(v_1_output.last_hidden_state, l_outputs.last_hidden_state)
print(updated_vis_states.shape)  # torch.Size([1, 2040, 768])，已融合语言语义

torch.Size([1, 2040, 768])


In [10]:
Qwen3_5VisionMLP = sys.modules[model.__class__.__module__].Qwen3_5VisionMLP

class Qwen3_5TextToVisionBlock(GradientCheckpointingLayer):
    """
    Text-to-Vision Cross-Attention Block.
    
    Allows vision tokens to attend to text tokens via cross-attention,
    followed by a vision-side MLP. Follows the pre-norm residual 
    architecture of Qwen3_5VisionBlock.
    
    Input:
        vision_hidden_states: [vis_seq, vis_hidden_size] or [batch, vis_seq, vis_hidden_size]
        lang_hidden_states:   [batch, text_seq, lang_hidden_size]
    
    Output:
        updated vision_hidden_states: same shape as input
    """
    
    def __init__(
        self,
        config,  # Qwen3_5VisionConfig or Qwen3_5Config
        lang_config=None,  # Optional: Qwen3_5TextConfig for language-side dims
        attn_implementation: str = "sdpa",
    ) -> None:
        super().__init__()
        
        # === Extract config parameters ===
        vis_hidden_size = config.hidden_size
        rms_norm_eps = getattr(config, "rms_norm_eps", 1e-6)
        
        # Language-side dims: prefer lang_config, fallback to config or defaults
        lang_hidden_size = getattr(lang_config, "hidden_size", 1024) if lang_config else getattr(config, "lang_hidden_size", 1024)
        num_attention_heads = getattr(config, "num_attention_heads", 12)
        num_key_value_heads = getattr(config, "num_key_value_heads", 4)
        attention_bias = getattr(config, "attention_bias", False)
        
        # === Cross-Attention Branch (Text → Vision) ===
        # Pre-norm for vision query (Qwen3.5 style RMSNorm)
        self.vision_norm1 = Qwen3_5RMSNorm(vis_hidden_size, eps=rms_norm_eps)
        
        # Cross-attention: Q from vision, K/V from text
        self.cross_attn = Qwen3_5LangToVisionCrossAttention(
            vis_hidden_size=vis_hidden_size,
            lang_hidden_size=lang_hidden_size,
            num_attention_heads=num_attention_heads,
            num_key_value_heads=num_key_value_heads,
            rms_norm_eps=rms_norm_eps,
            attention_bias=attention_bias,
        )
        
        # Post-norm after cross-attn residual connection
        self.vision_norm2 = Qwen3_5RMSNorm(vis_hidden_size, eps=rms_norm_eps)
        
        # === MLP Branch (Vision-side FFN) ===
        # Pre-norm for MLP
        self.vision_norm3 = Qwen3_5RMSNorm(vis_hidden_size, eps=rms_norm_eps)
        
        # Vision MLP (reusing Qwen3_5VisionMLP from the codebase)
        self.vision_mlp = Qwen3_5VisionMLP(config)

    def forward(
        self,
        vision_hidden_states: torch.Tensor,      # [vis_seq, vis_hidden] or [batch, vis_seq, vis_hidden]
        lang_hidden_states: torch.Tensor,        # [batch, text_seq, lang_hidden]
        attention_mask: torch.Tensor | None = None,
        **kwargs,
    ) -> torch.Tensor:
        # ─── Cross-Attention: Vision attends to Text ─────────────────────────
        residual = vision_hidden_states
        vision_normed = self.vision_norm1(vision_hidden_states)
        
        attn_output = self.cross_attn(
            vis_hidden_states=vision_normed,
            lang_hidden_states=lang_hidden_states,
            attention_mask=attention_mask,
        )
        # Residual + post-norm (Qwen3.5 pattern)
        vision_hidden_states = self.vision_norm2(residual + attn_output)
        
        # ─── MLP: Vision-side Feed-Forward ───────────────────────────────────
        residual = vision_hidden_states
        vision_normed = self.vision_norm3(vision_hidden_states)
        
        mlp_output = self.vision_mlp(vision_normed)
        # Residual connection (standard Transformer pattern)
        vision_hidden_states = residual + mlp_output
        
        return vision_hidden_states

In [40]:
# 假设 config 是 Qwen3_5VisionConfig 或 Qwen3_5Config
l_to_v_block = Qwen3_5TextToVisionBlock(config=model.model.visual.config)
l_to_v_block.to(device=model.device, dtype=model.dtype)

updated_vision = l_to_v_block(
    vision_hidden_states=v_1_output.last_hidden_state,
    lang_hidden_states=l_outputs.last_hidden_state
)  # → [2040, 768]
updated_vision.shape

torch.Size([1, 2040, 768])

## Pixel-Shuffle + MLP

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Tuple, List

class ShuffleMLPStage(nn.Module):
    """单层 Pixel-Unshuffle + 瓶颈MLP（兼容 Batch 维度）"""
    def __init__(self, in_channels: int, merge_size: int, hidden_size: int, use_postshuffle_norm: bool = True):
        super().__init__()
        self.merge_size = merge_size
        self.expanded_channels = in_channels * (merge_size ** 2)
        self.use_postshuffle_norm = use_postshuffle_norm

        self.norm = nn.LayerNorm(self.expanded_channels, eps=1e-6) if use_postshuffle_norm else nn.Identity()
        self.linear_fc1 = nn.Linear(self.expanded_channels, hidden_size)
        self.act_fn = nn.GELU()
        self.linear_fc2 = nn.Linear(hidden_size, hidden_size)

    def forward(self, x: torch.Tensor, h: int, w: int) -> Tuple[torch.Tensor, int, int]:
        # 🔧 兼容 2D (N, C) 和 3D (B, N, C) 输入
        if x.ndim == 2:
            x = x.unsqueeze(0)
        b, n, c = x.shape
        assert n == h * w, f"Token数量 {n} 与空间尺寸 {h}x{w} 不匹配"

        # 转为 4D 图像格式: (B, C, H, W)
        x = x.reshape(b, h, w, c).permute(0, 3, 1, 2)

        # 自适应边界填充（保证 H/W 能被 merge_size 整除）
        pad_h = (self.merge_size - h % self.merge_size) % self.merge_size
        pad_w = (self.merge_size - w % self.merge_size) % self.merge_size
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode='replicate')
            h, w = h + pad_h, w + pad_w

        # Pixel-Unshuffle 下采样
        x = F.pixel_unshuffle(x, self.merge_size)
        _, c_new, h_new, w_new = x.shape
        
        # 转回序列格式: (B, N_new, C_new)
        x = x.permute(0, 2, 3, 1).reshape(b, -1, c_new)

        # 归一化 + 瓶颈 MLP
        x = self.norm(x)
        x = self.linear_fc2(self.act_fn(self.linear_fc1(x)))
        return x, h_new, w_new


class VisionTokenCompressor(nn.Module):
    """多层 {Pixel-Unshuffle + Bottleneck MLP} 压缩器（自动处理 Batch 维度）"""
    def __init__(
        self, 
        in_hidden: int = 768,
        num_tokens: int = 2040,
        spatial_shape: Tuple[int, int] = (34, 60),
        merge_sizes: List[int] = (2, 2, 2),
        out_hidden: int = 768,
        use_postshuffle_norm: bool = True
    ):
        super().__init__()
        self.in_hidden = in_hidden
        self.spatial_shape = spatial_shape
        self.out_hidden = out_hidden
        self.stages = nn.ModuleList()

        for r in merge_sizes:
            self.stages.append(ShuffleMLPStage(in_hidden, r, in_hidden, use_postshuffle_norm))

        self.final_proj = nn.Linear(in_hidden, out_hidden)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 记录原始维度，输出时自动对齐
        is_2d = (x.ndim == 2)
        if is_2d:
            x = x.unsqueeze(0)

        h, w = self.spatial_shape
        for stage in self.stages:
            x, h, w = stage(x, h, w)
            
        x = self.final_proj(x)
        return x.squeeze(0) if is_2d else x

In [25]:
compressor = VisionTokenCompressor(
        in_hidden=768,
        num_tokens=2040,
        spatial_shape=(34, 60),
        merge_sizes=(2, 2, 2, 2, 2),  # 
        out_hidden=1024,
        use_postshuffle_norm=True
    ).to(device=model.device, dtype=model.dtype)

compressed_x = compressor(v_1_output.last_hidden_state)
compressed_x.shape

torch.Size([1, 4, 1024])

In [ ]:
act_head = VisionTokenCompressor(
        in_hidden=1024,
        num_tokens=4,
        spatial_shape=(2, 2),
        merge_sizes=(2, 2),  # 
        out_hidden=29,
        use_postshuffle_norm=True
    ).to(device=model.device, dtype=model.dtype)

compressed_x_1 = act_compressor(compressed_x)
compressed_x_1.shape

torch.Size([1, 1, 29])

## test reasoning speed and memory consumption

In [38]:
# manufacture-构造960x540的Image
from PIL import Image
img = Image.open("微信图片_20260327210311_589_2.jpg")
if img.mode != 'RGB':
    img = img.convert('RGB')
img = img.resize((960, 540), Image.LANCZOS)
visual_input = processor.image_processor(images=[img]).to(device)
video_input = [visual_input for _ in range(8*30)]
init_inputs_embeds = torch.randn(1, 1, 1024, device=model.device, dtype=model.dtype)

v_to_l_block = Qwen3_5VisionToTextBlock(
    model.config.text_config, 
vis_hidden_size=768).to(model.device, dtype=model.dtype)
l_to_v_block = Qwen3_5TextToVisionBlock(config=model.model.visual.config)
l_to_v_block.to(device=model.device, dtype=model.dtype)
compressor = VisionTokenCompressor(
        in_hidden=768,
        num_tokens=2040,
        spatial_shape=(34, 60),
        merge_sizes=(2, 2, 2, 2),  # 3层下采样：Token数约除以 4^3 = 64
        out_hidden=1024,
        use_postshuffle_norm=True
    ).to(device=model.device, dtype=model.dtype)

In [22]:
print(visual_input.pixel_values.shape)

torch.Size([2040, 1536])


In [24]:
for v in video_input:
    print(v.pixel_values.shape)

torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])
torch.Size([2040, 1536])


In [66]:
print(v_output.last_hidden_state.shape)

torch.Size([1, 2040, 768])


In [39]:
text_seq=init_inputs_embeds
with torch.no_grad():
    i=0
    for visual_input in video_input:
        v_output = model.model.visual.visual_stage_1_forward(
            hidden_states=visual_input.pixel_values, 
            grid_thw=visual_input.image_grid_thw, 
            stop_layer_idx=3, 
        )
        l_outputs = model.model.language_model.language_stage_1_forward(text_seq, depth=1)
        l_outputs  = v_to_l_block(
            lang_hidden_states=l_outputs.last_hidden_state,
            vis_hidden_states=v_output.last_hidden_state
        )

        v_output = model.model.visual.visual_stage_2_forward(
                hidden_states=v_output.last_hidden_state,
                start_layer=4,
                end_layer=7,
                grid_thw=visual_input.image_grid_thw
        )
        l_outputs = model.model.language_model.language_stage_2_forward(
            hidden_states=l_outputs,
            start_layer=1,  # 对应第2层
            end_layer=3     # 对应第4层
        )
        l_outputs  = v_to_l_block(
            lang_hidden_states=l_outputs .last_hidden_state,
            vis_hidden_states=v_output.last_hidden_state
        )
        l_outputs = model.model.language_model.language_stage_2_forward(
            hidden_states=l_outputs,
            start_layer=4,  # 对应第5层
            end_layer=23     # 对应第24层
        )
        v_output = l_to_v_block(
            vision_hidden_states=v_output.last_hidden_state,
            lang_hidden_states=l_outputs.last_hidden_state
        )
        v_output = model.model.visual.visual_stage_2_forward(
                hidden_states=v_output,
                start_layer=8,
                end_layer=11,
                grid_thw=visual_input.image_grid_thw
        )
        compressed_v = compressor(v_output.last_hidden_state)
        text_seq=torch.cat((text_seq, compressed_v), dim=1)
        print(f"{i}  显存占用: {torch.cuda.memory_allocated(device)/1024**3:.3f} GB", end='\r')

OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 15.70 GiB of which 82.94 MiB is free. Process 20537 has 15.61 GiB memory in use. Of the allocated memory 14.96 GiB is allocated by PyTorch, and 515.21 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [23]:
v_output.last_hidden_state.shape

torch.Size([1, 8480, 768])

In [53]:
compressed_v.unsqueeze(0).shape

torch.Size([1, 4, 1024])

In [13]:
torch.cat((compressed_v.unsqueeze(0),init_inputs_embeds), dim=1).shape

torch.Size([1, 964, 1024])

In [47]:
compressed_v.shape

torch.Size([1, 4, 768])

## packaging

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
from transformers.cache_utils import Cache, DynamicCache
from transformers.modeling_outputs import CausalLMOutputWithPast, BaseModelOutputWithPast
from transformers.modeling_utils import PreTrainedModel
Qwen3_5Config = sys.modules[model.__class__.__module__].Qwen3_5Config
Qwen3_5Model = sys.modules[model.__class__.__module__].Qwen3_5Model
Qwen3_5RMSNorm = sys.modules[model.__class__.__module__].Qwen3_5RMSNorm
# 假设以下自定义类与 monkey-patch 方法已在当前环境中加载
# from modeling_reqwen3_5 import (
#     visual_stage_1_forward, visual_stage_2_forward,
#     language_stage_1_forward, language_stage_2_forward,
#     Qwen3_5VisionToTextBlock, Qwen3_5TextToVisionBlock, VisionTokenCompressor
# )

class Qwen3_5StreamingForConditionalGeneration(PreTrainedModel):
    """流式视觉-语言交错推理模型"""
    
    # ✅ 添加这些类属性以支持各种注意力实现
    config_class = Qwen3_5Config
    base_model_prefix = "model"
    supports_gradient_checkpointing = True
    _no_split_modules = ["Qwen3_5DecoderLayer", "Qwen3_5VisionBlock"]
    _skip_keys_device_placement = "past_key_values"
    _supports_flash_attn = True
    _supports_sdpa = True
    _supports_flex_attn = True
    _is_stateful = True  # ✅ 支持 KV Cache 的关键属性
    
    # 可选：忽略加载时不需要的权重
    _keys_to_ignore_on_load_unexpected = [r"^mtp.*"]

    def __init__(self, config, *args, **kwargs):
        # ✅ 确保配置中的 _attn_implementation 被正确传递
        if "attn_implementation" in kwargs:
            config._attn_implementation_internal = kwargs.pop("attn_implementation")
        super().__init__(config, *args, **kwargs)
        # 基础模型组件
        self.model = Qwen3_5Model(config)
        self.lm_head = nn.Linear(config.text_config.hidden_size, config.text_config.vocab_size, bias=False)

        # 流式交互模块初始化
        self.v_to_l_block = Qwen3_5VisionToTextBlock(
            config.text_config, vis_hidden_size=768
        )
        self.l_to_v_block = Qwen3_5TextToVisionBlock(
            config.vision_config, lang_config=config.text_config
        )
        self.compressor = VisionTokenCompressor(
            in_hidden=768,
            num_tokens=2040,
            spatial_shape=(34, 60),
            merge_sizes=(2, 2, 2, 2, 2),
            out_hidden=config.text_config.hidden_size,
            use_postshuffle_norm=True
        )

        # 🔗 绑定流式前向方法（若外部未 patch 则在此绑定）
        '''self.model.model.visual.visual_stage_1_forward = torch.nn.MethodType(visual_stage_1_forward, self.model.model.visual)
        self.model.model.visual.visual_stage_2_forward = torch.nn.MethodType(visual_stage_2_forward, self.model.model.visual)
        self.model.model.language_model.language_stage_1_forward = torch.nn.MethodType(language_stage_1_forward, self.model.model.language_model)
        self.model.model.language_model.language_stage_2_forward = torch.nn.MethodType(language_stage_2_forward, self.model.model.language_model)'''

        self.post_init()

    def forward(
    self,
    inputs_embeds: Optional[torch.Tensor] = None,
    prev_hidden_states: Optional[torch.Tensor] = None,
    pixel_values: Optional[torch.Tensor] = None,      # ⚠️ 仅用于特征提取，不传给 self.model()
    image_grid_thw: Optional[torch.Tensor] = None,
    past_key_values: Optional[Cache] = None,
    use_cache: bool = True,
    attention_mask: Optional[torch.Tensor] = None,
    compute_logits: bool = True,
    return_dict: bool = False,
    ):
        # 1️⃣ 状态路由
        if inputs_embeds is not None:
            current_hidden = inputs_embeds
        else:
            if prev_hidden_states is None:
                raise ValueError("非首帧必须提供 prev_hidden_states")
            current_hidden = prev_hidden_states

        device, dtype = current_hidden.device, current_hidden.dtype
        batch_size, seq_len, _ = current_hidden.shape

        # 2️⃣ 初始化 KV Cache（全局共享）
        if use_cache and past_key_values is None:
            past_key_values = DynamicCache(config=self.config.text_config)

        # 3️⃣ 🔥 关键：手动提取视觉特征（不调用 self.model.get_image_features）
        v_out1 = self.model.visual.visual_stage_1_forward(
            hidden_states=pixel_values, 
            grid_thw=image_grid_thw, 
            stop_layer_idx=3
        )

        # 4️⃣ 语言 Stage 1（第 1 层）
        l_out1 = self.model.language_model.language_stage_1_forward(
            inputs_embeds=current_hidden,
            depth=1,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=use_cache
        )

        # 5️⃣ V→L 跨模态注入
        l_out1_hidden = self.v_to_l_block(
            lang_hidden_states=l_out1.last_hidden_state,
            vis_hidden_states=v_out1.last_hidden_state
        )

        # 6️⃣ 视觉 Stage 2 (4~7) + 语言 Stage 2 (2~4)
        v_out2 = self.model.visual.visual_stage_2_forward(
            hidden_states=v_out1.last_hidden_state,
            start_layer=4, end_layer=7,
            grid_thw=image_grid_thw
        )
        l_out2 = self.model.language_model.language_stage_2_forward(
            hidden_states=l_out1_hidden,
            start_layer=1, end_layer=3,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=use_cache
        )
        l_out2_hidden = self.v_to_l_block(
            lang_hidden_states=l_out2.last_hidden_state,
            vis_hidden_states=v_out2.last_hidden_state
        )

        # 7️⃣ 语言深度推理 (5~24) + L→V 反馈
        l_out3 = self.model.language_model.language_stage_2_forward(
            hidden_states=l_out2_hidden,
            start_layer=4, end_layer=23,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=use_cache
        )
        v_out3 = self.l_to_v_block(
            vision_hidden_states=v_out2.last_hidden_state,
            lang_hidden_states=l_out3.last_hidden_state
        )

        # 8️⃣ 视觉收尾 (8~11) + 压缩
        v_out4 = self.model.visual.visual_stage_2_forward(
            hidden_states=v_out3,
            start_layer=8, end_layer=11,
            grid_thw=image_grid_thw
        )
        compressed_v = self.compressor(v_out4.last_hidden_state)  # [B, new_seq, D]

        # 9️⃣ 拼接新序列
        new_hidden = torch.cat([current_hidden, compressed_v], dim=1)

        # 🔟 更新 attention_mask（若存在）
        if attention_mask is not None:
            pad_len = new_hidden.shape[1] - attention_mask.shape[1]
            if pad_len > 0:
                attention_mask = F.pad(attention_mask, (0, pad_len), value=1)

        # 1️⃣1️⃣ 计算 logits（可选）
        logits = self.lm_head(new_hidden[:, -1:, :]) if compute_logits else None

        if return_dict:
            return CausalLMOutputWithPast(
                logits=compressed_v,    #logits
                past_key_values=past_key_values if use_cache else None,
                last_hidden_state=new_hidden,
            )
        return new_hidden, past_key_values

In [44]:
remodel=Qwen3_5StreamingForConditionalGeneration(model.model.config, attn_implementation="eager")
remodel.model.visual.visual_stage_1_forward = types.MethodType(visual_stage_1_forward, remodel.model.visual)
remodel.model.visual.visual_stage_2_forward = types.MethodType(visual_stage_2_forward, remodel.model.visual)
remodel.model.language_model.language_stage_1_forward = types.MethodType(language_stage_1_forward, remodel.model.language_model)
remodel.model.language_model.language_stage_2_forward = types.MethodType(language_stage_2_forward, remodel.model.language_model)
remodel.to(model.device, dtype=model.dtype)

KeyboardInterrupt: 

In [13]:
from PIL import Image
img = Image.open("微信图片_20260327210311_589_2.jpg")
if img.mode != 'RGB':
    img = img.convert('RGB')
img = img.resize((960, 540), Image.LANCZOS)
visual_input = processor.image_processor(images=[img]).to(device)
video_input = [visual_input for _ in range(8*30)]
init_inputs_embeds = torch.randn(1, 1, 1024, device=model.device, dtype=model.dtype)

In [69]:
visual_input

{'pixel_values': tensor([[ 0.2549,  0.2549,  0.2549,  ...,  0.8039,  0.8039,  0.8039],
        [ 0.2549,  0.2549,  0.2706,  ...,  0.8118,  0.8118,  0.8118],
        [ 0.2471,  0.2471,  0.2471,  ...,  0.8196,  0.8196,  0.8196],
        ...,
        [-0.2627, -0.2471, -0.2471,  ..., -0.7490, -0.7804, -0.7882],
        [-0.4588, -0.4196, -0.3569,  ..., -0.5529, -0.5373, -0.4980],
        [-0.4980, -0.5294, -0.5529,  ..., -0.5059, -0.4980, -0.4902]],
       device='cuda:0'), 'image_grid_thw': tensor([[ 1, 34, 60]], device='cuda:0')}

In [22]:
outputs = remodel(
    inputs_embeds=init_inputs_embeds,
    pixel_values=visual_input.pixel_values,
    image_grid_thw=visual_input.image_grid_thw,
    use_cache=True,
    return_dict = True
)

TypeError: CausalLMOutputWithPast.__init__() got an unexpected keyword argument 'last_hidden_state'

In [21]:
outputs

(tensor([[[ 0.1128,  0.9883,  1.0938,  ...,  0.8008,  1.7031, -0.1934],
          [ 0.1226, -0.1104, -0.1172,  ..., -0.2188, -0.0845, -0.0850],
          [ 0.0383,  0.0067, -0.0742,  ..., -0.2383, -0.0280, -0.0199],
          ...,
          [ 0.0640, -0.0496, -0.0928,  ..., -0.4473, -0.1416,  0.0669],
          [ 0.0309, -0.0277, -0.0339,  ..., -0.3828, -0.2090, -0.0625],
          [-0.0466, -0.0688, -0.0376,  ..., -0.4297, -0.1738, -0.1138]]],
        device='cuda:0', dtype=torch.bfloat16, grad_fn=<CatBackward0>),
 DynamicCache(layers=[LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttentionLayer, DynamicLayer, LinearAttentionLayer, LinearAttentionLayer, LinearAttenti

In [15]:
prev_hidden = outputs[0]
cache = outputs[1]

In [16]:
outputs = remodel(
        prev_hidden_states=prev_hidden,  # ⬅️ 替代 inputs_embeds
        pixel_values=visual_input.pixel_values,
        image_grid_thw=visual_input.image_grid_thw,
        past_key_values=cache,
        use_cache=True
    )
prev_hidden = outputs[0]
cache = outputs[1]

RuntimeError: The expanded size of the tensor (14) must match the existing size (27) at non-singleton dimension 3.  Target sizes: [1, 8, 13, 14].  Tensor sizes: [1, 1, 13, 27]

In [33]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple
from transformers.cache_utils import Cache, DynamicCache
from transformers.modeling_outputs import CausalLMOutputWithPast, BaseModelOutputWithPast
from transformers.modeling_utils import PreTrainedModel
Qwen3_5Config = sys.modules[model.__class__.__module__].Qwen3_5Config
Qwen3_5Model = sys.modules[model.__class__.__module__].Qwen3_5Model
Qwen3_5RMSNorm = sys.modules[model.__class__.__module__].Qwen3_5RMSNorm

class Qwen3_5StreamingForPPO(PreTrainedModel):
    """
    流式视觉-语言模型 + PPO 强化学习头
    输出: action_logits [B, 4, act_dim], value_estimates [B, 1]
    """
    config_class = Qwen3_5Config
    base_model_prefix = "model"
    supports_gradient_checkpointing = True
    _no_split_modules = ["Qwen3_5DecoderLayer", "Qwen3_5VisionBlock"]
    _skip_keys_device_placement = "past_key_values"
    _supports_flash_attn = True
    _supports_sdpa = True
    _supports_flex_attn = True
    _is_stateful = True

    def __init__(self, config, act_dim: int = 29, *args, **kwargs):
        # 处理 attn_implementation 参数
        attn_impl = kwargs.pop("attn_implementation", None)
        if attn_impl is not None:
            config._attn_implementation_internal = attn_impl
        
        super().__init__(config, *args, **kwargs)
        
        # 基础模型组件
        self.model = Qwen3_5Model(config)
        # 🔥 移除 lm_head（PPO 不需要语言建模 logits）

        # 流式交互模块
        self.v_to_l_block = Qwen3_5VisionToTextBlock(
            config.text_config, vis_hidden_size=768
        )
        self.l_to_v_block = Qwen3_5TextToVisionBlock(
            config.vision_config, lang_config=config.text_config
        )
        self.compressor = VisionTokenCompressor(
            in_hidden=768,
            num_tokens=2040,
            spatial_shape=(34, 60),
            merge_sizes=(2, 2, 2, 2, 2),
            out_hidden=config.text_config.hidden_size,  # 1024
            use_postshuffle_norm=True
        )

        # 🎯 PPO 专用头：接收 compressed_v [B, 4, 1024]
        self.act_head = VisionTokenCompressor(
            in_hidden=config.text_config.hidden_size,  # 1024
            num_tokens=4,
            spatial_shape=(2, 2),
            merge_sizes=(2, 2),
            out_hidden=act_dim,  # 29: 动作空间大小
            use_postshuffle_norm=True
        )
        self.value_head = VisionTokenCompressor(
            in_hidden=config.text_config.hidden_size,  # 1024
            num_tokens=4,
            spatial_shape=(2, 2),
            merge_sizes=(2, 2),
            out_hidden=1,  # 标量价值估计
            use_postshuffle_norm=True
        )

        self.post_init()
    def reset_streaming_state(self):
        """
        重置流式状态与 KV Cache，适用于 PPO 新 Episode/轨迹开始。
        清理 prev_hidden_states、past_key_values 及所有累积的线性注意力隐状态。
        """
        self._prev_hidden = None
        self._past_key_values = None
        # DynamicCache 会在下次 use_cache=True 时自动重建
        return self
    def forward(
        self,
        inputs_embeds: Optional[torch.Tensor] = None,
        prev_hidden_states: Optional[torch.Tensor] = None,
        pixel_values: Optional[torch.Tensor] = None,
        image_grid_thw: Optional[torch.Tensor] = None,
        past_key_values: Optional[Cache] = None,
        use_cache: bool = True,
        attention_mask: Optional[torch.Tensor] = None,
        return_rl_outputs: bool = True,  # 🔥 控制是否计算 RL 头
        return_dict: bool = True,
    ):
        # === 1. 状态路由 ===
        if inputs_embeds is not None:
            current_hidden = inputs_embeds
        else:
            if prev_hidden_states is None:
                raise ValueError("非首帧必须提供 prev_hidden_states")
            current_hidden = prev_hidden_states

        device, dtype = current_hidden.device, current_hidden.dtype
        batch_size, seq_len, _ = current_hidden.shape

        # === 2. 初始化 KV Cache ===
        if use_cache and past_key_values is None:
            past_key_values = DynamicCache(config=self.config.text_config)

        # === 3~11. 视觉-语言交错推理（与原逻辑一致）===
        # [视觉 Stage 1] → [语言 Stage 1] → [V→L] → [视觉 Stage 2] → [语言 Stage 2] 
        # → [V→L] → [语言深度推理] → [L→V] → [视觉收尾]
        
        v_out1 = self.model.visual.visual_stage_1_forward(
            hidden_states=pixel_values, grid_thw=image_grid_thw, stop_layer_idx=3
        )
        l_out1 = self.model.language_model.language_stage_1_forward(
            inputs_embeds=current_hidden, depth=1,
            attention_mask=attention_mask, past_key_values=past_key_values, use_cache=use_cache
        )
        l_out1_hidden = self.v_to_l_block(
            lang_hidden_states=l_out1.last_hidden_state,
            vis_hidden_states=v_out1.last_hidden_state
        )
        v_out2 = self.model.visual.visual_stage_2_forward(
            hidden_states=v_out1.last_hidden_state, start_layer=4, end_layer=7, grid_thw=image_grid_thw
        )
        l_out2 = self.model.language_model.language_stage_2_forward(
            hidden_states=l_out1_hidden, start_layer=1, end_layer=3,
            attention_mask=attention_mask, past_key_values=past_key_values, use_cache=use_cache
        )
        l_out2_hidden = self.v_to_l_block(
            lang_hidden_states=l_out2.last_hidden_state,
            vis_hidden_states=v_out2.last_hidden_state
        )
        l_out3 = self.model.language_model.language_stage_2_forward(
            hidden_states=l_out2_hidden, start_layer=4, end_layer=23,
            attention_mask=attention_mask, past_key_values=past_key_values, use_cache=use_cache
        )
        v_out3 = self.l_to_v_block(
            vision_hidden_states=v_out2.last_hidden_state,
            lang_hidden_states=l_out3.last_hidden_state
        )
        v_out4 = self.model.visual.visual_stage_2_forward(
            hidden_states=v_out3, start_layer=8, end_layer=11, grid_thw=image_grid_thw
        )

        # === 12. 视觉 Token 压缩 ===
        compressed_v = self.compressor(v_out4.last_hidden_state)  # [B, 4, 1024]
        if compressed_v.dim() == 2:
            compressed_v = compressed_v.unsqueeze(0)

        # === 13. 序列拼接（维持流式状态）===
        new_hidden = torch.cat([current_hidden, compressed_v], dim=1)
        if attention_mask is not None:
            pad_len = new_hidden.shape[1] - attention_mask.shape[1]
            if pad_len > 0:
                attention_mask = F.pad(attention_mask, (0, pad_len), value=1)

        # === 14. 🔥 PPO 头计算（可选）===
        action_logits, value_estimates = None, None
        if return_rl_outputs:
            # act_head: [B, 4, 1024] → [B, 4, act_dim]
            action_logits = self.act_head(compressed_v)  # [B, 4, 29]
            # value_head: [B, 4, 1024] → [B, 4, 1] → 取 mean 得标量价值
            value_raw = self.value_head(compressed_v)    # [B, 4, 1]
            value_estimates = value_raw.mean(dim=1, keepdim=True)  # [B, 1]

        # === 15. 返回标准化输出 ===
        if return_dict:
            from transformers.modeling_outputs import ModelOutput
            return ModelOutput(
                last_hidden_state=new_hidden if use_cache else None,
                past_key_values=past_key_values if use_cache else None,
                action_logits=action_logits,      # 🔥 [B, 4, act_dim]
                value_estimates=value_estimates,  # 🔥 [B, 1]
                compressed_v=compressed_v,        # 可选：用于调试/可视化
            )
        return new_hidden, past_key_values, action_logits, value_estimates

In [35]:
def language_stage_1_forward(
    self,
    inputs_embeds: torch.Tensor,
    depth: int,
    attention_mask: torch.Tensor | None = None,  # ⚠️ 流式模式下此参数将被安全忽略
    past_key_values: DynamicCache | None = None,
    use_cache: bool = True
) -> "Qwen3_5ModelOutputWithPast":
    assert 0 < depth <= len(self.layers), f"depth 必须在 1 到 {len(self.layers)} 之间"
    if use_cache and past_key_values is None:
        past_key_values = DynamicCache(config=self.config)

    device = inputs_embeds.device
    batch_size, seq_len, _ = inputs_embeds.shape
    past_seen = past_key_values.get_seq_length() if past_key_values is not None else 0

    # 🔑 1. 构造 4D Position IDs
    position_ids = torch.arange(past_seen, past_seen + seq_len, device=device)
    position_ids = position_ids.view(1, batch_size, -1).expand(4, batch_size, -1)  # (4, B, S)
    text_position_ids = position_ids[0]      # 供 DecoderLayer 使用
    rope_position_ids = position_ids[1:]     # 供 RotaryEmbedding 使用 (3, B, S)

    # 🔑 2. 手动构造增量 Causal Mask (彻底避开 create_causal_mask 的广播坑)
    # SDPA 期望形状: [Batch, 1, Query_Seq, Key_Total]
    row_idx = torch.arange(seq_len, device=device).unsqueeze(1)      # [S, 1]
    col_idx = torch.arange(past_seen + seq_len, device=device)       # [K_total]
    causal_mask = (col_idx <= row_idx + past_seen).unsqueeze(0).unsqueeze(0)  # [1, 1, S, K_total]

    # 🔑 3. Linear Attention Mask (Prefill 阶段需保留，Decode 阶段置 None)
    linear_attn_mask = None if past_seen > 0 else causal_mask

    # 🔑 4. 计算 RoPE
    position_embeddings = self.rotary_emb(inputs_embeds, rope_position_ids)

    # 🔑 5. 逐层 Forward
    hidden_states = inputs_embeds
    for i in range(depth):
        layer_type = self.config.layer_types[i]
        layer_mask = linear_attn_mask if layer_type == "linear_attention" else causal_mask

        hidden_states = self.layers[i](
            hidden_states,
            position_embeddings=position_embeddings,
            attention_mask=layer_mask,
            position_ids=text_position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
        )

    return Qwen3_5ModelOutputWithPast(
        last_hidden_state=hidden_states,
        past_key_values=past_key_values if use_cache else None,
    )

def language_stage_2_forward(
    self,
    hidden_states: torch.Tensor,
    start_layer: int = 2,
    end_layer: int = 8,
    attention_mask: torch.Tensor | None = None,  # ⚠️ 流式模式下此参数将被安全忽略
    past_key_values=None,
    use_cache: bool = True,
    **kwargs
) -> "Qwen3_5ModelOutputWithPast":
    if use_cache and past_key_values is None:
        past_key_values = DynamicCache(config=self.config)

    device = hidden_states.device
    batch_size, seq_len, _ = hidden_states.shape
    past_seen = past_key_values.get_seq_length() if past_key_values is not None else 0
    total_len = past_seen + seq_len

    # 🔑 1. 构造 4D Position IDs
    position_ids = torch.arange(past_seen, total_len, device=device)
    position_ids = position_ids.view(1, batch_size, -1).expand(4, batch_size, -1)
    text_position_ids = position_ids[0]
    rope_position_ids = position_ids[1:]

    # 🔑 2. 手动构造增量 Causal Mask
    row_idx = torch.arange(seq_len, device=device).unsqueeze(1)
    col_idx = torch.arange(total_len, device=device)
    causal_mask = (col_idx <= row_idx + past_seen).unsqueeze(0).unsqueeze(0)

    # 🔑 3. Linear Attention Mask
    linear_attn_mask = None if past_seen > 0 else causal_mask

    # 🔑 4. 计算 RoPE
    position_embeddings = self.rotary_emb(hidden_states, rope_position_ids)

    # 🔑 5. 逐层 Forward
    for i in range(start_layer, end_layer + 1):
        layer_type = self.config.layer_types[i]
        layer_mask = linear_attn_mask if layer_type == "linear_attention" else causal_mask

        hidden_states = self.layers[i](
            hidden_states=hidden_states,
            position_embeddings=position_embeddings,
            attention_mask=layer_mask,
            position_ids=text_position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            **kwargs
        )

    return Qwen3_5ModelOutputWithPast(
        last_hidden_state=hidden_states,
        past_key_values=past_key_values if use_cache else None,
    )

In [36]:
remodel=Qwen3_5StreamingForPPO(model.model.config, attn_implementation="eager")
remodel.model.visual.visual_stage_1_forward = types.MethodType(visual_stage_1_forward, remodel.model.visual)
remodel.model.visual.visual_stage_2_forward = types.MethodType(visual_stage_2_forward, remodel.model.visual)
remodel.model.language_model.language_stage_1_forward = types.MethodType(language_stage_1_forward, remodel.model.language_model)
remodel.model.language_model.language_stage_2_forward = types.MethodType(language_stage_2_forward, remodel.model.language_model)
remodel.to(model.device, dtype=model.dtype)

Qwen3_5StreamingForPPO(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 768)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-11): 12 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=768, out_features=3072, bias=True)
            (linear_fc2): Linear(in_features=3072, out_features=768, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
      (merger): Qwe

In [16]:
from PIL import Image
img = Image.open("微信图片_20260327210311_589_2.jpg")
if img.mode != 'RGB':
    img = img.convert('RGB')
img = img.resize((960, 540), Image.LANCZOS)
visual_input = processor.image_processor(images=[img]).to(device)
video_input = [visual_input for _ in range(8*30)]
init_inputs_embeds = torch.randn(1, 1, 1024, device=model.device, dtype=model.dtype)

In [37]:
outputs = remodel(
    inputs_embeds=init_inputs_embeds,
    pixel_values=visual_input.pixel_values,
    image_grid_thw=visual_input.image_grid_thw,
    use_cache=True,
    return_rl_outputs=True  # 🔥 启用 RL 头
)

RuntimeError: The expanded size of the tensor (1) must match the existing size (2) at non-singleton dimension 3.  Target sizes: [1, 8, 1, 1].  Tensor sizes: [1, 1, 1, 2]

In [27]:
prev_hidden = outputs.last_hidden_state
cache = outputs.past_key_values

In [28]:
outputs = remodel(
        prev_hidden_states=prev_hidden,
        pixel_values=visual_input.pixel_values,
        image_grid_thw=visual_input.image_grid_thw,
        past_key_values=cache,
        use_cache=True,
        return_rl_outputs=True
    )

RuntimeError: The expanded size of the tensor (6) must match the existing size (11) at non-singleton dimension 3.  Target sizes: [1, 8, 5, 6].  Tensor sizes: [1, 1, 5, 11]